# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-azeemi/fatima-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice: Random Forest Classifier

* **Problem Framing:** Predicting whether a content page will experience a performance decline (`trend_direction == 'down'`) is a binary classification task.
* **Why Random Forest Fits Our Lane:**
  1. SEO metrics (like impressions and content age) have heavy tails and non-linear interactions. Random Forest handles these naturally without requiring delicate scaling.
  2. It avoids the severe overfitting common in single decision trees while providing robust feature importance scores to explain what drives page decay.
* **Baseline to Beat:** Week-4 Rule-Based Baseline (`STALE_HIGH_IMPRESSIONS` heuristic rule).

In [37]:
import pandas as pd
import numpy as np
import os, sys

# Setup repository path safely for Google Colab if needed
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load dataset locally from repo storage
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Prepare target variable and baseline feature
df['target'] = (df['trend_direction'] == 'down').astype(int)
df['is_stale_high_imp'] = ((df['content_age_days'] > 180) & (df['impressions_90d'] > 500)).astype(int)

print("=== DATASET LOADED SUCCESSFULLY LOCALLY ===")
print(f"Total Rows: {len(df):,}")
print(f"Target Distribution (Decline Rate Base Rate):")
print(df['target'].value_counts(normalize=True).round(4) * 100)

# Check missing values count and percentage for our active features
active_features = [
    'word_count',
    'content_age_days',
    'impressions_90d',
    'avg_position',
    'ctr',
    'search_volume'
]

existing_features = [col for col in active_features if col in df.columns]

missing_summary = pd.DataFrame({
    'Missing Count': df[existing_features].isnull().sum(),
    'Missing Percentage (%)': (df[existing_features].isnull().mean() * 100).round(2)
})

print("\n=== DATASET MISSING VALUES AUDIT ===")
print(missing_summary.to_string())

# Impute missing entries cleanly using median strategy across all active features
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
df_imputed = df.copy()
df_imputed[existing_features] = imputer.fit_transform(df[existing_features])

print("\n=== MISSING VALUES AFTER IMPUTATION ===")
print(df_imputed[existing_features].isnull().sum().to_string())

=== DATASET LOADED SUCCESSFULLY LOCALLY ===
Total Rows: 30,000
Target Distribution (Decline Rate Base Rate):
target
1    54.21
0    45.79
Name: proportion, dtype: float64

=== DATASET MISSING VALUES AUDIT ===
                  Missing Count  Missing Percentage (%)
word_count                 7699                   25.66
content_age_days              0                    0.00
impressions_90d               0                    0.00
avg_position                  0                    0.00
ctr                           0                    0.00
search_volume              2468                    8.23

=== MISSING VALUES AFTER IMPUTATION ===
word_count          0
content_age_days    0
impressions_90d     0
avg_position        0
ctr                 0
search_volume       0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Honest Split Strategy: Stratified Train-Test Split (80/20)

* **Design:** We use an 80/20 stratified train-test split (`random_state=42`) based on our target variable (`trend_direction == 'down'`).
* **Why it's honest:**
  1. Prevents Data Leakage: Training data is strictly isolated from the 20% unseen test set.
  2. Preserves Base Rates: Stratification maintains the ~54.21% decline rate across both splits.
  3. Fair Evaluation: The baseline rule and the Random Forest model are evaluated on the exact same test records.

In [38]:
from sklearn.model_selection import train_test_split

# 1. Use all 6 active audit features consistently
feature_cols = [
    'word_count',
    'content_age_days',
    'impressions_90d',
    'avg_position',
    'ctr',
    'search_volume'
]
available_features = [col for col in feature_cols if col in df.columns]

X = df[available_features]
y = df['target']

# 2. 80/20 Stratified Split
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, df.index, test_size=0.20, random_state=42, stratify=y
)

# Extract test dataframe for baseline comparison
test_df = df.loc[test_idx].copy()

print("=== SPLIT DESIGN EXECUTED SUCCESSFULLY ===")
print(f"Training set size: {len(X_train):,} rows ({len(X_train)/len(df)*100:.0f}%)")
print(f"Test size:     {len(X_test):,} rows ({len(X_test)/len(df)*100:.0f}%)")
print(f"Train decline rate: {y_train.mean()*100:.2f}%")
print(f"Test decline rate:  {y_test.mean()*100:.2f}%")

=== SPLIT DESIGN EXECUTED SUCCESSFULLY ===
Training set size: 24,000 rows (80%)
Test size:     6,000 rows (20%)
Train decline rate: 54.21%
Test decline rate:  54.20%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model Training & Honest Comparison Table

We trained a **Random Forest Classifier** (`n_estimators=100, max_depth=6, random_state=42`) using the 80% training split. We then compared it against:
1. **Base Rate Baseline:** Always predicting page decline.
2. **Week-4 Rule Baseline:** The heuristic `STALE_HIGH_IMPRESSIONS` flag.

All models and baselines are evaluated on the exact same 20% unseen test split using standard classification metrics.

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.impute import SimpleImputer

# 1. Split FIRST to prevent any data leakage during imputation
X_train_raw, X_test_raw, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, df.index, test_size=0.20, random_state=42, stratify=y
)

# 2. Impute missing values SEPARATELY using Training Median (Honest Leakage-Free Way)
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=available_features)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_raw), columns=available_features)

# Extract test dataframe for baseline comparison
test_df_all = df.loc[test_idx].copy()
test_df_all['baseline_pred'] = test_df_all['is_stale_high_imp']

# 3. Train Random Forest Model (Optimized slightly for better generalization)
rf_model_all = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_split=10,
    random_state=42
)
rf_model_all.fit(X_train_imputed, y_train)

rf_pred_all = rf_model_all.predict(X_test_imputed)
rf_prob_all = rf_model_all.predict_proba(X_test_imputed)[:, 1]

# 4. Evaluation Metrics Function
def get_metrics(y_true, y_pred, y_prob=None):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0),
        roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    ]

base_rate_pred = np.ones(len(y_test))
base_metrics = get_metrics(y_test, base_rate_pred)
baseline_metrics = get_metrics(y_test, test_df_all['baseline_pred'])
rf_metrics_all = get_metrics(y_test, rf_pred_all, rf_prob_all)

# Comparison Table
comparison_df = pd.DataFrame({
    'Model / Strategy': ['Base Rate (Always Down)', 'Week-4 Rule Baseline', 'Random Forest (Optimized)'],
    'Accuracy': [base_metrics[0], baseline_metrics[0], rf_metrics_all[0]],
    'Precision': [base_metrics[1], baseline_metrics[1], rf_metrics_all[1]],
    'Recall': [base_metrics[2], baseline_metrics[2], rf_metrics_all[2]],
    'F1-Score': [base_metrics[3], baseline_metrics[3], rf_metrics_all[3]],
    'ROC-AUC': [base_metrics[4], baseline_metrics[4], rf_metrics_all[4]]
})

print("=== HONEST MODEL COMPARISON TABLE (OPTIMIZED) ===")
print(comparison_df.to_string(index=False))

=== HONEST MODEL COMPARISON TABLE (OPTIMIZED) ===
         Model / Strategy  Accuracy  Precision   Recall  F1-Score  ROC-AUC
  Base Rate (Always Down)  0.542000   0.542000 1.000000  0.702983      NaN
     Week-4 Rule Baseline  0.481833   0.535948 0.327798  0.406793      NaN
Random Forest (Optimized)  0.676833   0.664660 0.814883  0.732145 0.737389


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error Analysis & Feature Interpretations

* **Top Features:** Feature importances show that `content_age_days`, `impressions_90d`, and traffic metrics drive the model's decisions.
* **Where the Model Fails (Errors):**
  1. False Positives: Predicts decline for high-traffic or older pages that experienced temporary fluctuations but eventually recovered.
  2. False Negatives: Misses quiet, subtle traffic drops on short or low-impression pages where statistical signals are weak.
* **Practical Takeaway:** The learned model offers reliable decision-support over rigid rules, though SEO performance inherently contains high real-world noise.

In [40]:
# Feature Importances & Error Cases Visual Verification
importances = pd.DataFrame({
    'Feature': available_features,
    'Importance': rf_model_all.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("=== FEATURE IMPORTANCES ===")
print(importances.to_string(index=False))

test_df_all['rf_pred'] = rf_pred_all
test_df_all['is_error'] = test_df_all['target'] != test_df_all['rf_pred']

print("\n=== SAMPLE CONCRETE ERROR CASES ===")
error_cols = [col for col in ['content_age_days', 'impressions_90d', 'word_count', 'target', 'rf_pred'] if col in test_df_all.columns]
print(test_df_all[test_df_all['is_error']][error_cols].head(3).to_string())

=== FEATURE IMPORTANCES ===
         Feature  Importance
 impressions_90d    0.343082
    avg_position    0.233345
content_age_days    0.209154
      word_count    0.092688
             ctr    0.071464
   search_volume    0.050268

=== SAMPLE CONCRETE ERROR CASES ===
       content_age_days  impressions_90d  word_count  target  rf_pred
2110                223              511      6466.0       0        1
27890               463              881         NaN       1        0
25967               502              373         NaN       1        0


## 5. Charts + exports

*Save comparison / importance charts to `work/outputs/charts/` (and mirror under `outputs/charts/`) for the report.*

In [ ]:
# Save model charts + metrics for outputs/model_report.md
from pathlib import Path
import json
import sys

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return cand
    return here

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import simple_svg_bar_chart

work_charts = ROOT / "work" / "outputs" / "charts"
root_charts = ROOT / "outputs" / "charts"
work_charts.mkdir(parents=True, exist_ok=True)
root_charts.mkdir(parents=True, exist_ok=True)
(ROOT / "work" / "outputs").mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(parents=True, exist_ok=True)

def _save(name, title, labels, values, color="#2C5F2D"):
    for folder in (work_charts, root_charts):
        simple_svg_bar_chart(title, labels, values, folder / name, color=color)

_save(
    "model_comparison.svg",
    "Test-set comparison (Accuracy / F1 / ROC-AUC x100)",
    ["Base Acc", "Rule Acc", "RF Acc", "Base F1", "Rule F1", "RF F1", "RF ROC-AUC"],
    [
        float(comparison_df.loc[0, "Accuracy"]) * 100,
        float(comparison_df.loc[1, "Accuracy"]) * 100,
        float(comparison_df.loc[2, "Accuracy"]) * 100,
        float(comparison_df.loc[0, "F1-Score"]) * 100,
        float(comparison_df.loc[1, "F1-Score"]) * 100,
        float(comparison_df.loc[2, "F1-Score"]) * 100,
        float(comparison_df.loc[2, "ROC-AUC"]) * 100,
    ],
)
_save(
    "top_feature_importance.svg",
    "Random Forest feature importance (6-feature model)",
    importances["Feature"].tolist(),
    (importances["Importance"] * 100).tolist(),
    color="#4A6FA5",
)

metrics_payload = {
    "comparison": comparison_df.to_dict(orient="records"),
    "feature_importance": importances.to_dict(orient="records"),
    "split": {"train": int(len(X_train_imputed)), "test": int(len(X_test_imputed))},
}
(ROOT / "work" / "outputs" / "w05_model_metrics.json").write_text(
    json.dumps(metrics_payload, indent=2), encoding="utf-8"
)
print("Saved charts -> work/outputs/charts/ and outputs/charts/")
print("Saved metrics -> work/outputs/w05_model_metrics.json")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.